<a href="https://colab.research.google.com/github/atikhasan007/Machine-Learning/blob/main/project_c1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# import all necessary library

In [1]:
# Basic Python Libraries
import os
import random
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data Preprocessing
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

#  Statistical Analysis
from scipy import stats

# Deep Learning - PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)




In [2]:
# Device Configuration
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

PyTorch version: 2.11.0+cu128
Device: cuda


**Dataset load**

In [3]:
import pandas as pd

df_2025 = pd.read_csv(
    "/content/Alice_Springs_2025.csv",
    engine="python",
    on_bad_lines="skip"
)

df_2026 = pd.read_csv(
    "/content/Alice_Springs_2026.csv",
    engine="python",
    on_bad_lines="skip"
)

print("2025 shape:", df_2025.shape)
print("2026 shape:", df_2026.shape)

2025 shape: (105391, 210)
2026 shape: (12090, 210)


**Basic Check 2025 data**

In [4]:

df_2025.shape


(105391, 210)

In [5]:
df_2025.columns


Index(['timestamp',
       '205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received',
       '205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average',
       '205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power',
       '239_DKA_Totals_BESS_Reactive_Power',
       '239_DKA_Totals_BESS_Active_Power',
       '239_DKA_Totals_BESS_Apparent_Power',
       '239_DKA_Totals_BESS_State_of_Charge',
       '242_DKA_Totals_Grid_Reactive_Power',
       '242_DKA_Totals_Grid_Active_Power',
       ...
       '62_DKA_MasterMeter2_THD_Voltage_Average',
       '101_DKA_WeatherStation_Wind_Speed',
       '101_DKA_WeatherStation_Weather_Temperature_Celsius',
       '101_DKA_WeatherStation_Weather_Relative_Humidity',
       '101_DKA_WeatherStation_Global_Horizontal_Radiation',
       '101_DKA_WeatherStation_Diffuse_Horizontal_Radiation',
       '101_DKA_WeatherStation_Wind_Direction',
       '101_DKA_WeatherStation_Weather_Daily_Rainfall',
       '101_DKA_WeatherStation_Radiation_Globa

In [6]:
df_2025.isnull().sum()

,0
timestamp,0
205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received,105391
205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average,105391
205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power,105391
239_DKA_Totals_BESS_Reactive_Power,80
...,...
101_DKA_WeatherStation_Diffuse_Horizontal_Radiation,6267
101_DKA_WeatherStation_Wind_Direction,6267
101_DKA_WeatherStation_Weather_Daily_Rainfall,6267
101_DKA_WeatherStation_Radiation_Global_Tilted,1592


**Basic check 2026 data**

In [7]:
df_2026.shape

(12090, 210)

In [8]:
df_2025.columns

Index(['timestamp',
       '205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received',
       '205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average',
       '205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power',
       '239_DKA_Totals_BESS_Reactive_Power',
       '239_DKA_Totals_BESS_Active_Power',
       '239_DKA_Totals_BESS_Apparent_Power',
       '239_DKA_Totals_BESS_State_of_Charge',
       '242_DKA_Totals_Grid_Reactive_Power',
       '242_DKA_Totals_Grid_Active_Power',
       ...
       '62_DKA_MasterMeter2_THD_Voltage_Average',
       '101_DKA_WeatherStation_Wind_Speed',
       '101_DKA_WeatherStation_Weather_Temperature_Celsius',
       '101_DKA_WeatherStation_Weather_Relative_Humidity',
       '101_DKA_WeatherStation_Global_Horizontal_Radiation',
       '101_DKA_WeatherStation_Diffuse_Horizontal_Radiation',
       '101_DKA_WeatherStation_Wind_Direction',
       '101_DKA_WeatherStation_Weather_Daily_Rainfall',
       '101_DKA_WeatherStation_Radiation_Globa

In [9]:
df_2025.isnull().sum()

,0
timestamp,0
205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received,105391
205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average,105391
205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power,105391
239_DKA_Totals_BESS_Reactive_Power,80
...,...
101_DKA_WeatherStation_Diffuse_Horizontal_Radiation,6267
101_DKA_WeatherStation_Wind_Direction,6267
101_DKA_WeatherStation_Weather_Daily_Rainfall,6267
101_DKA_WeatherStation_Radiation_Global_Tilted,1592


**Convert timestamp**
The timestamp column in a dataset is often stored in string/text format initially. Converting it to datetime format makes it easier to work with time-series data.

In [10]:
df_2025["timestamp"] = pd.to_datetime(df_2025["timestamp"])
df_2026["timestamp"] = pd.to_datetime(df_2026["timestamp"])

**Dataset marge**

In [11]:
# Merge datasets
df = pd.concat(
    [df_2025, df_2026],
    axis=0,
    ignore_index=True
)

In [12]:
print("shape of df_2025 =", df_2025.shape)
print("shape of df_2026 =", df_2026.shape)
print("shape of df =", df.shape)


shape of df_2025 = (105391, 210)
shape of df_2026 = (12090, 210)
shape of df = (117481, 406)


**Sort data by timestamp**
Sorted data by timestamp to maintain chronological order and ensure correct lag, rolling features, and time-series splitting

In [13]:

df = df.sort_values("timestamp").reset_index(drop=True)

In [14]:
# Check duplicate rows
duplicate_rows = df.duplicated().sum()

print("Total duplicate rows:", duplicate_rows)

Total duplicate rows: 0


In [15]:
# Check total missing values
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 25813247


In [16]:
# Missing values per column
missing_values = df.isnull().sum()

print(missing_values[missing_values > 0])

205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received    117481
205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average               117481
205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power                        117481
239_DKA_Totals_BESS_Reactive_Power                                          12170
239_DKA_Totals_BESS_Active_Power                                            12170
                                                                            ...  
101_Diffuse_Horizontal_Radiation                                           105436
101_Wind_Direction                                                         105436
101_Weather_Daily_Rainfall                                                 105436
101_Radiation_Global_Tilted                                                105483
101_Radiation_Diffuse_Tilted                                               105483
Length: 405, dtype: int64


In [17]:
# Missing values summary
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df)) * 100
})

# Only columns having missing values
missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing %", ascending=False)

print(missing_summary)

                                                    Missing Count   Missing %
205_Archived_DKA_M15_BPhase_UMG_QCells_Active_E...         117481  100.000000
205_Archived_DKA_M15_BPhase_UMG_QCells_Active_P...         117481  100.000000
205_Archived_DKA_M15_BPhase_UMG_QCells_Current_...         117481  100.000000
101_Wind_Speed                                             117481  100.000000
99_DKA_M4_C_Phase_Active_Energy_Delivered_Received         117481  100.000000
...                                                           ...         ...
100_DKA_M1_A_Phase_Current_Phase_Average                      257    0.218759
100_DKA_M1_A_Phase_Performance_Ratio                          257    0.218759
103_DKA_M1_B_Phase_Current_Phase_Average                      257    0.218759
100_DKA_M1_A_Phase_Active_Power                               257    0.218759
103_DKA_M1_B_Phase_Active_Power                               257    0.218759

[405 rows x 2 columns]


In [18]:
# Find columns with 100% missing values
all_missing_cols = df.columns[df.isnull().all()]

print("100% missing columns:", len(all_missing_cols))
print(all_missing_cols.tolist())

100% missing columns: 43
['205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Energy_Delivered_Received', '205_Archived_DKA_M15_BPhase_UMG_QCells_Current_Phase_Average', '205_Archived_DKA_M15_BPhase_UMG_QCells_Active_Power', '54_DKA_M15_C_Phase_Active_Energy_Delivered_Received', '54_DKA_M15_C_Phase_Current_Phase_Average', '54_DKA_M15_C_Phase_Active_Power', '54_DKA_M15_C_Phase_Performance_Ratio', '57_DKA_M16_A_Phase_Active_Energy_Delivered_Received', '57_DKA_M16_A_Phase_Current_Phase_Average', '57_DKA_M16_A_Phase_Active_Power', '57_DKA_M16_A_Phase_Performance_Ratio', '77_DKA_M18_B_Phase_Active_Energy_Delivered_Received', '77_DKA_M18_B_Phase_Current_Phase_Average', '77_DKA_M18_B_Phase_Active_Power', '77_DKA_M18_B_Phase_Performance_Ratio', '99_DKA_M4_C_Phase_Active_Energy_Delivered_Received', '99_DKA_M4_C_Phase_Current_Phase_Average', '99_DKA_M4_C_Phase_Active_Power', '99_DKA_M4_C_Phase_Performance_Ratio', '102_DKA_M9_3_Phase_Active_Energy_Delivered_Received', '102_DKA_M9_3_Phase_Current_Phase

In [19]:
# Remove columns where all values are missing
df = df.dropna(axis=1, how="all")

print("New dataset shape:", df.shape)

New dataset shape: (117481, 363)


In [20]:
# again missing value check
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df)) * 100
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing %", ascending=False)

print(missing_summary)

                                          Missing Count  Missing %
101_Radiation_Diffuse_Tilted                     105483  89.787285
101_Radiation_Global_Tilted                      105483  89.787285
64_Current_Phase_Average                         105438  89.748981
64_Active_Energy_Delivered_Received              105438  89.748981
66_Active_Power                                  105438  89.748981
...                                                 ...        ...
100_DKA_M1_A_Phase_Performance_Ratio                257   0.218759
103_DKA_M1_B_Phase_Current_Phase_Average            257   0.218759
100_DKA_M1_A_Phase_Active_Power                     257   0.218759
103_DKA_M1_B_Phase_Active_Power                     257   0.218759
100_DKA_M1_A_Phase_Current_Phase_Average            257   0.218759

[362 rows x 2 columns]


In [21]:
# Calculate missing percentage
missing_percentage = df.isnull().mean() * 100

# Find columns with more than 50% missing
high_missing_cols = missing_percentage[missing_percentage > 50]

print("Columns with >50% missing:", len(high_missing_cols))
print(high_missing_cols)

Columns with >50% missing: 176
239_Reactive_Power                  89.736213
239_Active_Power                    89.736213
239_Apparent_Power                  89.736213
239_State_of_Charge                 89.735361
242_Reactive_Power                  89.735361
                                      ...    
101_Diffuse_Horizontal_Radiation    89.747278
101_Wind_Direction                  89.747278
101_Weather_Daily_Rainfall          89.747278
101_Radiation_Global_Tilted         89.787285
101_Radiation_Diffuse_Tilted        89.787285
Length: 176, dtype: float64


In [22]:
# Remove columns with more than 50% missing values
df = df.drop(columns=high_missing_cols.index)

print("Dataset shape after removing high-missing columns:", df.shape)

Dataset shape after removing high-missing columns: (117481, 187)


In [23]:
missing = df.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False))

101_DKA_WeatherStation_Diffuse_Horizontal_Radiation    18357
101_DKA_WeatherStation_Global_Horizontal_Radiation     18357
101_DKA_WeatherStation_Weather_Daily_Rainfall          18357
101_DKA_WeatherStation_Wind_Direction                  18357
101_DKA_WeatherStation_Weather_Relative_Humidity       18357
                                                       ...  
100_DKA_M1_A_Phase_Performance_Ratio                     257
103_DKA_M1_B_Phase_Active_Power                          257
100_DKA_M1_A_Phase_Active_Power                          257
103_DKA_M1_B_Phase_Current_Phase_Average                 257
100_DKA_M1_A_Phase_Current_Phase_Average                 257
Length: 186, dtype: int64


In [24]:
# Checked missing values in each column and calculated their count and percentage
missing = df.isnull().sum()

missing_info = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": (missing / len(df)) * 100
})

missing_info = missing_info[
    missing_info["missing_count"] > 0
].sort_values(
    "missing_percentage",
    ascending=False
)

print(missing_info)

                                                    missing_count  \
101_DKA_WeatherStation_Diffuse_Horizontal_Radia...          18357   
101_DKA_WeatherStation_Global_Horizontal_Radiation          18357   
101_DKA_WeatherStation_Weather_Daily_Rainfall               18357   
101_DKA_WeatherStation_Wind_Direction                       18357   
101_DKA_WeatherStation_Weather_Relative_Humidity            18357   
...                                                           ...   
100_DKA_M1_A_Phase_Performance_Ratio                          257   
103_DKA_M1_B_Phase_Active_Power                               257   
100_DKA_M1_A_Phase_Active_Power                               257   
103_DKA_M1_B_Phase_Current_Phase_Average                      257   
100_DKA_M1_A_Phase_Current_Phase_Average                      257   

                                                    missing_percentage  
101_DKA_WeatherStation_Diffuse_Horizontal_Radia...           15.625505  
101_DKA_WeatherStation_Gl

In [25]:

# Select numerical columns
numeric_cols = df.select_dtypes(include="number").columns

# Interpolate small missing gaps
df[numeric_cols] = df[numeric_cols].interpolate(
    method="linear",
    limit=12
)

print("Interpolation completed.")

Interpolation completed.


In [26]:
# Rechecked the dataset after cleaning to verify that no missing values remain
missing_after = df.isnull().sum()

missing_after = missing_after[
    missing_after > 0
].sort_values(ascending=False)

print(missing_after)

101_DKA_WeatherStation_Diffuse_Horizontal_Radiation    17909
101_DKA_WeatherStation_Global_Horizontal_Radiation     17909
101_DKA_WeatherStation_Weather_Daily_Rainfall          17909
101_DKA_WeatherStation_Wind_Direction                  17909
101_DKA_WeatherStation_Weather_Relative_Humidity       17909
                                                       ...  
103_DKA_M1_B_Phase_Current_Phase_Average                  65
105_DKA_M1_C_Phase_Current_Phase_Average                  65
105_DKA_M1_C_Phase_Active_Energy_Delivered_Received       65
100_DKA_M1_A_Phase_Current_Phase_Average                  65
100_DKA_M1_A_Phase_Active_Energy_Delivered_Received       65
Length: 186, dtype: int64


In [27]:
for col in df.columns:
    if "power" in col.lower():
        print(col)

239_DKA_Totals_BESS_Reactive_Power
239_DKA_Totals_BESS_Active_Power
239_DKA_Totals_BESS_Apparent_Power
242_DKA_Totals_Grid_Reactive_Power
242_DKA_Totals_Grid_Active_Power
242_DKA_Totals_Grid_Apparent_Power
241_DKA_Totals_PV_Reactive_Power
241_DKA_Totals_PV_Active_Power
241_DKA_Totals_PV_Apparent_Power
240_DKA_Totals_Site_Demand_Reactive_Power
240_DKA_Totals_Site_Demand_Active_Power
240_DKA_Totals_Site_Demand_Apparent_Power
100_DKA_M1_A_Phase_Active_Power
103_DKA_M1_B_Phase_Active_Power
105_DKA_M1_C_Phase_Active_Power
97_DKA_M10_B_C_Phases_Active_Power
78_DKA_M11_3_Phase_Active_Power
61_DKA_M15_A_Phase_Active_Power
72_DKA_M15_B_Phase_Active_Power
212_DKA_M15_C_Phase_II_Active_Power
213_DKA_M16_A_Phase_II_Active_Power
66_DKA_M16_B_Phase_Active_Power
52_DKA_M16_C_Phase_Active_Power
63_DKA_M17_A_Phase_Active_Power
64_DKA_M17_B_Phase_Active_Power
58_DKA_M17_C_Phase_Active_Power
60_DKA_M18_A_Phase_Active_Power
214_DKA_M18_B_Phase_II_Active_Power
74_DKA_M18_C_Phase_Active_Power
73_DKA_M19_A_P

In [28]:
# Define target column
target_col = "241_DKA_Totals_PV_Active_Power"

print("Target column:", target_col)
print("Target missing values:", df[target_col].isnull().sum())
print(
    "Target missing percentage:",
    df[target_col].isnull().mean() * 100
)

Target column: 241_DKA_Totals_PV_Active_Power
Target missing values: 11791
Target missing percentage: 10.036516543100586


In [29]:
# Checked the timestamps where the target variable has missing values
print(
    df[df[target_col].isnull()][
        ["timestamp", target_col]
    ].head(20)
)

                 timestamp  241_DKA_Totals_PV_Active_Power
105690 2026-01-02 00:55:00                             NaN
105691 2026-01-02 01:00:00                             NaN
105692 2026-01-02 01:05:00                             NaN
105693 2026-01-02 01:10:00                             NaN
105694 2026-01-02 01:15:00                             NaN
105695 2026-01-02 01:20:00                             NaN
105696 2026-01-02 01:25:00                             NaN
105697 2026-01-02 01:30:00                             NaN
105698 2026-01-02 01:35:00                             NaN
105699 2026-01-02 01:40:00                             NaN
105700 2026-01-02 01:45:00                             NaN
105701 2026-01-02 01:50:00                             NaN
105702 2026-01-02 01:55:00                             NaN
105703 2026-01-02 02:00:00                             NaN
105704 2026-01-02 02:05:00                             NaN
105705 2026-01-02 02:10:00                             N

In [30]:
# It checks where the target column has missing values and how many missing values there are.
target_col = "241_DKA_Totals_PV_Active_Power"

missing_target = df[df[target_col].isna()]

print("First missing timestamp:",
      missing_target["timestamp"].min())

print("Last missing timestamp:",
      missing_target["timestamp"].max())

print("Total missing:",
      missing_target.shape[0])

First missing timestamp: 2026-01-02 00:55:00
Last missing timestamp: 2026-02-11 23:55:00
Total missing: 11791


In [31]:
# It checks the number of missing values in the target column for each year.
print(
    df.groupby(df["timestamp"].dt.year)[target_col]
      .apply(lambda x: x.isna().sum())
)

timestamp
2025        0
2026    11791
Name: 241_DKA_Totals_PV_Active_Power, dtype: int64


In [32]:
# Rename PV target column
df = df.rename(columns={
    "241_DKA_Totals_PV_Active_Power": "pv_power"
})

target_col = "pv_power"

print("Target:", target_col)
print("Missing target:", df[target_col].isna().sum())

Target: pv_power
Missing target: 11791


In [33]:
# Remove rows with missing PV power
df = df.dropna(subset=["pv_power"]).reset_index(drop=True)

print("Dataset shape after removing missing target rows:", df.shape)
print("Remaining missing target:", df["pv_power"].isna().sum())

Dataset shape after removing missing target rows: (105690, 187)
Remaining missing target: 0


In [34]:
# Sort by timestamp
df = df.sort_values("timestamp").reset_index(drop=True)

# Check time intervals
time_diff = df["timestamp"].diff()

print(time_diff.value_counts().head(10))

timestamp
0 days 00:05:00    105369
0 days 00:00:00       288
0 days 00:10:00        15
0 days 00:15:00         3
0 days 00:00:03         2
0 days 00:04:59         2
0 days 00:04:58         2
0 days 00:04:57         2
0 days 00:05:19         1
0 days 00:00:01         1
Name: count, dtype: int64


In [35]:
# It checks and displays the number of missing values in each column of the dataset.
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

101_DKA_WeatherStation_Weather_Temperature_Celsius       6118
101_DKA_WeatherStation_Weather_Daily_Rainfall            6118
101_DKA_WeatherStation_Wind_Direction                    6118
101_DKA_WeatherStation_Diffuse_Horizontal_Radiation      6118
101_DKA_WeatherStation_Global_Horizontal_Radiation       6118
                                                         ... 
87_DKA_M9_A_C_Phases_Active_Energy_Delivered_Received      65
68_DKA_M8_C_Phase_Active_Power                             65
68_DKA_M8_C_Phase_Current_Phase_Average                    65
68_DKA_M8_C_Phase_Active_Energy_Delivered_Received         65
96_DKA_MasterMeter1_THD_Voltage_Average                    65
Length: 173, dtype: int64


In [36]:

### It calculates the number and percentage of missing values for each column and displays the columns with missing values in descending order.


missing_info = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": df.isnull().mean() * 100
})

print(
    missing_info[
        missing_info["missing_count"] > 0
    ].sort_values("missing_percentage", ascending=False)
)

                                                    missing_count  \
101_DKA_WeatherStation_Weather_Temperature_Celsius           6118   
101_DKA_WeatherStation_Weather_Daily_Rainfall                6118   
101_DKA_WeatherStation_Wind_Direction                        6118   
101_DKA_WeatherStation_Diffuse_Horizontal_Radia...           6118   
101_DKA_WeatherStation_Global_Horizontal_Radiation           6118   
...                                                           ...   
87_DKA_M9_A_C_Phases_Active_Energy_Delivered_Re...             65   
68_DKA_M8_C_Phase_Active_Power                                 65   
68_DKA_M8_C_Phase_Current_Phase_Average                        65   
68_DKA_M8_C_Phase_Active_Energy_Delivered_Received             65   
96_DKA_MasterMeter1_THD_Voltage_Average                        65   

                                                    missing_percentage  
101_DKA_WeatherStation_Weather_Temperature_Celsius            5.788627  
101_DKA_WeatherStation_We

In [37]:
# check Pv power missing
print("PV power missing:", df["pv_power"].isna().sum())

PV power missing: 0


In [38]:
# Fill very small gaps using interpolation
numeric_cols = df.select_dtypes(include="number").columns

df[numeric_cols] = df[numeric_cols].interpolate(
    method="linear",
    limit=2
)

In [39]:
# It checks the number of missing values in the selected weather-related features.
weather_cols = [
    "101_DKA_WeatherStation_Weather_Daily_Rainfall",
    "101_DKA_WeatherStation_Weather_Temperature_Celsius",
    "101_DKA_WeatherStation_Weather_Relative_Humidity",
    "101_DKA_WeatherStation_Wind_Direction",
    "101_DKA_WeatherStation_Diffuse_Horizontal_Radiation",
    "101_DKA_WeatherStation_Global_Horizontal_Radiation"
]

print(df[weather_cols].isna().sum())

101_DKA_WeatherStation_Weather_Daily_Rainfall          6114
101_DKA_WeatherStation_Weather_Temperature_Celsius     6114
101_DKA_WeatherStation_Weather_Relative_Humidity       6114
101_DKA_WeatherStation_Wind_Direction                  6114
101_DKA_WeatherStation_Diffuse_Horizontal_Radiation    6114
101_DKA_WeatherStation_Global_Horizontal_Radiation     6114
dtype: int64


In [40]:
# It identifies the time range and total number of missing values for each weather feature
for col in weather_cols:
    missing_rows = df.loc[df[col].isna(), "timestamp"]

    print("\n", col)
    print("First:", missing_rows.min())
    print("Last :", missing_rows.max())
    print("Count:", missing_rows.count())


 101_DKA_WeatherStation_Weather_Daily_Rainfall
First: 2025-01-06 19:25:00
Last : 2025-11-03 00:25:00
Count: 6114

 101_DKA_WeatherStation_Weather_Temperature_Celsius
First: 2025-01-06 19:25:00
Last : 2025-11-03 00:25:00
Count: 6114

 101_DKA_WeatherStation_Weather_Relative_Humidity
First: 2025-01-06 19:25:00
Last : 2025-11-03 00:25:00
Count: 6114

 101_DKA_WeatherStation_Wind_Direction
First: 2025-01-06 19:25:00
Last : 2025-11-03 00:25:00
Count: 6114

 101_DKA_WeatherStation_Diffuse_Horizontal_Radiation
First: 2025-01-06 19:25:00
Last : 2025-11-03 00:25:00
Count: 6114

 101_DKA_WeatherStation_Global_Horizontal_Radiation
First: 2025-01-06 19:25:00
Last : 2025-11-03 00:25:00
Count: 6114


In [41]:
weather_cols = [
    "101_DKA_WeatherStation_Weather_Daily_Rainfall",
    "101_DKA_WeatherStation_Weather_Temperature_Celsius",
    "101_DKA_WeatherStation_Weather_Relative_Humidity",
    "101_DKA_WeatherStation_Wind_Direction",
    "101_DKA_WeatherStation_Diffuse_Horizontal_Radiation",
    "101_DKA_WeatherStation_Global_Horizontal_Radiation"
]

# Remove rows where weather data is missing
df = df.dropna(subset=weather_cols).reset_index(drop=True)

print("Dataset shape after removing weather-gap rows:", df.shape)

Dataset shape after removing weather-gap rows: (99576, 187)


In [42]:
missing = df.isnull().sum()

print(
    missing[missing > 0]
    .sort_values(ascending=False)
)

101_DKA_WeatherStation_Radiation_Diffuse_Tilted        1042
101_DKA_WeatherStation_Radiation_Global_Tilted         1042
61_DKA_M15_A_Phase_Active_Power                           1
61_DKA_M15_A_Phase_Current_Phase_Average                  1
72_DKA_M15_B_Phase_Active_Energy_Delivered_Received       1
                                                       ... 
62_DKA_MasterMeter2_Active_Power                          1
62_DKA_MasterMeter2_Average_Voltage_Line_to_Neutral       1
62_DKA_MasterMeter2_Power_Factor_Signed                   1
62_DKA_MasterMeter2_THD_Voltage_Average                   1
62_DKA_MasterMeter2_Frequency                             1
Length: 73, dtype: int64


In [43]:
# Check remaining missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Remaining missing values:")
print(missing)

Remaining missing values:
101_DKA_WeatherStation_Radiation_Diffuse_Tilted        1042
101_DKA_WeatherStation_Radiation_Global_Tilted         1042
61_DKA_M15_A_Phase_Active_Power                           1
61_DKA_M15_A_Phase_Current_Phase_Average                  1
72_DKA_M15_B_Phase_Active_Energy_Delivered_Received       1
                                                       ... 
62_DKA_MasterMeter2_Active_Power                          1
62_DKA_MasterMeter2_Average_Voltage_Line_to_Neutral       1
62_DKA_MasterMeter2_Power_Factor_Signed                   1
62_DKA_MasterMeter2_THD_Voltage_Average                   1
62_DKA_MasterMeter2_Frequency                             1
Length: 73, dtype: int64


In [44]:
# Fill only small remaining gaps
numeric_cols = df.select_dtypes(include="number").columns

df[numeric_cols] = (
    df[numeric_cols]
    .interpolate(method="linear", limit=2)
    .ffill(limit=2)
    .bfill(limit=2)
)

print("Total remaining missing values:",
      df.isnull().sum().sum())

Total remaining missing values: 2070


In [45]:
# Remove duplicate timestamps
duplicate_count = df["timestamp"].duplicated().sum()

print("Duplicate timestamps:", duplicate_count)

df = (
    df.drop_duplicates(subset=["timestamp"], keep="first")
      .sort_values("timestamp")
      .reset_index(drop=True)
)

print("Shape after removing duplicates:", df.shape)

Duplicate timestamps: 288
Shape after removing duplicates: (99288, 187)


In [46]:
# It checks the number of negative PV power values and identifies the maximum PV power value in the dataset.
print("Negative PV power:",
      (df["pv_power"] < 0).sum())

print("Maximum PV power:",
      df["pv_power"].max())

Negative PV power: 51128
Maximum PV power: 229.54742431641


In [47]:

#It extracts all negative PV power values and performs a statistical analysis of these anomalous observations.
negative_pv = df.loc[df["pv_power"] < 0, "pv_power"]

print("Negative PV count:", len(negative_pv))
print("Minimum PV power:", negative_pv.min())
print("Maximum negative PV power:", negative_pv.max())
print("\nNegative PV statistics:")
print(negative_pv.describe())

Negative PV count: 51128
Minimum PV power: -1.14537525177
Maximum negative PV power: -0.0003488101065158

Negative PV statistics:
count    51128.000000
mean        -0.614472
std          0.132799
min         -1.145375
25%         -0.726158
50%         -0.586591
75%         -0.495113
max         -0.000349
Name: pv_power, dtype: float64


In [48]:
# Convert negative PV power values to zero
df.loc[df["pv_power"] < 0, "pv_power"] = 0

print("Negative PV values after correction:",
      (df["pv_power"] < 0).sum())

print("Minimum PV power after correction:",
      df["pv_power"].min())

Negative PV values after correction: 0
Minimum PV power after correction: 0.0


In [49]:
print(df["pv_power"].describe())

count    99288.000000
mean        54.881004
std         73.390784
min          0.000000
25%          0.000000
50%          0.000000
75%        120.333382
max        229.547424
Name: pv_power, dtype: float64


In [50]:
print("Zero PV values:",
      (df["pv_power"] == 0).sum())

print("Positive PV values:",
      (df["pv_power"] > 0).sum())

Zero PV values: 51178
Positive PV values: 48110


In [51]:
# Make sure timestamp is datetime
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Sort chronologically
df = df.sort_values("timestamp").reset_index(drop=True)

print(df["timestamp"].head())
print(df["timestamp"].tail())

0   2025-01-01 00:00:00
1   2025-01-01 00:05:00
2   2025-01-01 00:10:00
3   2025-01-01 00:15:00
4   2025-01-01 00:20:00
Name: timestamp, dtype: datetime64[ns]
99283   2026-01-02 00:30:00
99284   2026-01-02 00:35:00
99285   2026-01-02 00:40:00
99286   2026-01-02 00:45:00
99287   2026-01-02 00:50:00
Name: timestamp, dtype: datetime64[ns]


In [52]:
# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [53]:
print("Shape:", df.shape)

Shape: (99288, 187)


In [54]:
print("Duplicate timestamps:", df["timestamp"].duplicated().sum())

Duplicate timestamps: 0


In [55]:
print("Missing PV:", df["pv_power"].isna().sum())
print("Negative PV:", (df["pv_power"] < 0).sum())
print("Zero PV:", (df["pv_power"] == 0).sum())
print("Maximum PV:", df["pv_power"].max())

Missing PV: 0
Negative PV: 0
Zero PV: 51178
Maximum PV: 229.54742431641


In [56]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": df.isna().mean() * 100
})

missing_summary = missing_summary[
    missing_summary["Missing Count"] > 0
].sort_values("Missing %", ascending=False)

print(missing_summary)

                                                 Missing Count  Missing %
101_DKA_WeatherStation_Radiation_Global_Tilted            1035   1.042422
101_DKA_WeatherStation_Radiation_Diffuse_Tilted           1035   1.042422


In [57]:
df = df.sort_values("timestamp").reset_index(drop=True)

df = df.set_index("timestamp")

In [58]:
# Missing values in the global tilted and diffuse tilted radiation features were filled using time-based interpolation.
radiation_cols = [
    "101_DKA_WeatherStation_Radiation_Global_Tilted",
    "101_DKA_WeatherStation_Radiation_Diffuse_Tilted"
]

df[radiation_cols] = df[radiation_cols].interpolate(
    method="time"
)

In [59]:
df[radiation_cols] = df[radiation_cols].ffill().bfill()

In [60]:
print(df[radiation_cols].isna().sum())

101_DKA_WeatherStation_Radiation_Global_Tilted     0
101_DKA_WeatherStation_Radiation_Diffuse_Tilted    0
dtype: int64


In [61]:
df = df.reset_index()

In [62]:
print("Total missing values:", df.isna().sum().sum())

Total missing values: 0


In [63]:
print("Duplicate timestamps:", df["timestamp"].duplicated().sum())

time_diff = df["timestamp"].diff().dropna()

print(time_diff.value_counts().head(10))

Duplicate timestamps: 0
timestamp
0 days 00:05:00     99253
0 days 00:10:00        15
0 days 00:15:00         3
0 days 00:04:59         2
0 days 00:04:58         2
0 days 00:04:57         2
0 days 00:00:03         2
0 days 00:05:19         1
0 days 00:00:01         1
21 days 00:20:00        1
Name: count, dtype: int64


In [64]:
print("Dataset shape:", df.shape)
print("Start:", df["timestamp"].min())
print("End:", df["timestamp"].max())

Dataset shape: (99288, 187)
Start: 2025-01-01 00:00:00
End: 2026-01-02 00:50:00


In [65]:
time_diff = df["timestamp"].diff()

large_gaps = df.loc[
    time_diff > pd.Timedelta(minutes=30),
    ["timestamp"]
].copy()

large_gaps["gap"] = time_diff[
    time_diff > pd.Timedelta(minutes=30)
].values

print(large_gaps)

                timestamp              gap
1673  2025-01-27 19:40:00 21 days 00:20:00
82011 2025-11-03 00:30:00  0 days 05:20:00


In [66]:
time_diff = df["timestamp"].diff()

print("5-minute gaps:",
      (time_diff == pd.Timedelta(minutes=5)).sum())

print("10-minute gaps:",
      (time_diff == pd.Timedelta(minutes=10)).sum())

print("Other gaps:",
      ((time_diff != pd.Timedelta(minutes=5)) &
       time_diff.notna()).sum())

5-minute gaps: 99253
10-minute gaps: 15
Other gaps: 34


**Feature Selection**

In [67]:
target = "pv_power"

X = df.drop(columns=["pv_power", "timestamp"])
y = df["pv_power"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (99288, 185)
Target: (99288,)


In [68]:
# Check features with only one unique value

nunique = X.nunique()

constant_features = nunique[nunique <= 1].index

print("Constant features:", len(constant_features))
print(constant_features.tolist())

Constant features: 0
[]


In [69]:
# Correlation of each feature with PV power

corr_with_target = X.corrwith(y).abs().sort_values(ascending=False)

print("Top 30 features correlated with pv_power:")
print(corr_with_target.head(30))

Top 30 features correlated with pv_power:
241_DKA_Totals_PV_Apparent_Power                  0.999923
101_DKA_WeatherStation_Radiation_Global_Tilted    0.977313
96_DKA_MasterMeter1_Active_Power                  0.972498
62_DKA_MasterMeter2_Active_Power                  0.971359
212_DKA_M15_C_Phase_II_Active_Power               0.971329
62_DKA_MasterMeter2_Current_Phase_Average         0.971145
212_DKA_M15_C_Phase_II_Current_Phase_Average      0.971026
213_DKA_M16_A_Phase_II_Active_Power               0.970954
58_DKA_M17_C_Phase_Active_Power                   0.970933
213_DKA_M16_A_Phase_II_Current_Phase_Average      0.970895
52_DKA_M16_C_Phase_Active_Power                   0.970892
55_DKA_M20_B_Phase_Active_Power                   0.970868
63_DKA_M17_A_Phase_Active_Power                   0.970852
96_DKA_MasterMeter1_Current_Phase_Average         0.970848
56_DKA_M20_A_Phase_Active_Power                   0.970800
214_DKA_M18_B_Phase_II_Active_Power               0.970789
56_DKA_M20_A_P

In [70]:
feature = "241_DKA_Totals_PV_Apparent_Power"

print(df[[feature, "pv_power"]].describe())

print(
    df[[feature, "pv_power"]]
    .corr()
)

       241_DKA_Totals_PV_Apparent_Power      pv_power
count                      99288.000000  99288.000000
mean                          56.752932     54.881004
std                           72.288554     73.390784
min                            0.000000      0.000000
25%                            3.035305      0.000000
50%                            3.444550      0.000000
75%                          120.466501    120.333382
max                          230.075287    229.547424
                                  241_DKA_Totals_PV_Apparent_Power  pv_power
241_DKA_Totals_PV_Apparent_Power                          1.000000  0.999923
pv_power                                                  0.999923  1.000000


In [71]:
feature = "241_DKA_Totals_PV_Apparent_Power"

diff = (
    df[feature] - df["pv_power"]
).abs()

print("Mean difference:", diff.mean())
print("Median difference:", diff.median())
print("Maximum difference:", diff.max())
print("Exact same values:", (diff == 0).sum())
print("Same percentage:", (diff == 0).mean() * 100)

Mean difference: 1.8719609512565611
Median difference: 2.6948392391205003
Maximum difference: 6.7185392379761
Exact same values: 179
Same percentage: 0.18028361936991377


In [72]:
# Remove highly target-coupled PV apparent power feature

leakage_feature = "241_DKA_Totals_PV_Apparent_Power"

X = X.drop(columns=[leakage_feature])

print("Features remaining:", X.shape[1])

Features remaining: 184


In [73]:
# Feature-to-feature correlation

corr_matrix = X.corr().abs()

upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper.stack()
    .sort_values(ascending=False)
)

high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.95]

print("Highly correlated feature pairs:", len(high_corr_pairs))
print(high_corr_pairs.head(50))

Highly correlated feature pairs: 3313
98_DKA_M8_B_Phase_Current_Phase_Average                  91_DKA_M9_B_Phase_Current_Phase_Average                    1.000000
52_DKA_M16_C_Phase_Active_Energy_Delivered_Received      214_DKA_M18_B_Phase_II_Active_Energy_Delivered_Received    0.999999
                                                         55_DKA_M20_B_Phase_Active_Energy_Delivered_Received        0.999999
214_DKA_M18_B_Phase_II_Active_Energy_Delivered_Received  84_DKA_M5_B_Phase_Active_Energy_Delivered_Received         0.999999
                                                         55_DKA_M20_B_Phase_Active_Energy_Delivered_Received        0.999999
56_DKA_M20_A_Phase_Active_Energy_Delivered_Received      84_DKA_M5_B_Phase_Active_Energy_Delivered_Received         0.999999
214_DKA_M18_B_Phase_II_Active_Energy_Delivered_Received  98_DKA_M8_B_Phase_Active_Energy_Delivered_Received         0.999999
56_DKA_M20_A_Phase_Active_Energy_Delivered_Received      98_DKA_M8_B_Phase_Active_Energ

In [74]:
import numpy as np

# Absolute correlation matrix
corr_matrix = X.corr().abs()

# Upper triangle
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# Highly correlated features
to_drop = [
    column
    for column in upper.columns
    if any(upper[column] > 0.995)
]

print("Features to remove:", len(to_drop))
print(to_drop)

Features to remove: 107
['100_DKA_M1_A_Phase_Active_Power', '103_DKA_M1_B_Phase_Active_Energy_Delivered_Received', '103_DKA_M1_B_Phase_Active_Power', '105_DKA_M1_C_Phase_Active_Energy_Delivered_Received', '105_DKA_M1_C_Phase_Active_Power', '97_DKA_M10_B_C_Phases_Active_Energy_Delivered_Received', '97_DKA_M10_B_C_Phases_Active_Power', '78_DKA_M11_3_Phase_Active_Energy_Delivered_Received', '78_DKA_M11_3_Phase_Current_Phase_Average', '78_DKA_M11_3_Phase_Active_Power', '61_DKA_M15_A_Phase_Active_Energy_Delivered_Received', '61_DKA_M15_A_Phase_Current_Phase_Average', '61_DKA_M15_A_Phase_Active_Power', '72_DKA_M15_B_Phase_Active_Power', '212_DKA_M15_C_Phase_II_Active_Energy_Delivered_Received', '212_DKA_M15_C_Phase_II_Current_Phase_Average', '212_DKA_M15_C_Phase_II_Active_Power', '213_DKA_M16_A_Phase_II_Active_Energy_Delivered_Received', '213_DKA_M16_A_Phase_II_Current_Phase_Average', '213_DKA_M16_A_Phase_II_Active_Power', '66_DKA_M16_B_Phase_Active_Energy_Delivered_Received', '66_DKA_M16_B_

In [75]:
X_reduced = X.drop(columns=to_drop)

print("Original features:", X.shape[1])
print("After correlation filtering:", X_reduced.shape[1])

Original features: 184
After correlation filtering: 77


**Feature Engineering**

In [76]:
df_fe = df[["timestamp", "pv_power"] + X_reduced.columns.tolist()].copy()

df_fe["timestamp"] = pd.to_datetime(df_fe["timestamp"])

df_fe = df_fe.sort_values("timestamp").reset_index(drop=True)

print(df_fe.shape)
print(df_fe[["timestamp", "pv_power"]].head())

(99288, 79)
            timestamp  pv_power
0 2025-01-01 00:00:00       0.0
1 2025-01-01 00:05:00       0.0
2 2025-01-01 00:10:00       0.0
3 2025-01-01 00:15:00       0.0
4 2025-01-01 00:20:00       0.0


In [77]:
# time features
df_fe["hour"] = df_fe["timestamp"].dt.hour
df_fe["minute"] = df_fe["timestamp"].dt.minute
df_fe["day_of_week"] = df_fe["timestamp"].dt.dayofweek
df_fe["day_of_year"] = df_fe["timestamp"].dt.dayofyear
df_fe["month"] = df_fe["timestamp"].dt.month

print(df_fe[
    ["timestamp", "hour", "minute", "day_of_week", "day_of_year", "month"]
].head(10))

            timestamp  hour  minute  day_of_week  day_of_year  month
0 2025-01-01 00:00:00     0       0            2            1      1
1 2025-01-01 00:05:00     0       5            2            1      1
2 2025-01-01 00:10:00     0      10            2            1      1
3 2025-01-01 00:15:00     0      15            2            1      1
4 2025-01-01 00:20:00     0      20            2            1      1
5 2025-01-01 00:25:00     0      25            2            1      1
6 2025-01-01 00:30:00     0      30            2            1      1
7 2025-01-01 00:35:00     0      35            2            1      1
8 2025-01-01 00:40:00     0      40            2            1      1
9 2025-01-01 00:45:00     0      45            2            1      1


In [78]:
# Cyclic Time Features
import numpy as np

# Hour + minute → total minutes of day
df_fe["minute_of_day"] = (
    df_fe["hour"] * 60 + df_fe["minute"]
)

# Daily cycle
df_fe["hour_sin"] = np.sin(
    2 * np.pi * df_fe["minute_of_day"] / 1440
)

df_fe["hour_cos"] = np.cos(
    2 * np.pi * df_fe["minute_of_day"] / 1440
)

# Weekly cycle
df_fe["dow_sin"] = np.sin(
    2 * np.pi * df_fe["day_of_week"] / 7
)

df_fe["dow_cos"] = np.cos(
    2 * np.pi * df_fe["day_of_week"] / 7
)

# Yearly cycle
df_fe["doy_sin"] = np.sin(
    2 * np.pi * df_fe["day_of_year"] / 365
)

df_fe["doy_cos"] = np.cos(
    2 * np.pi * df_fe["day_of_year"] / 365
)

print(df_fe.shape)

print(
    df_fe[
        [
            "timestamp",
            "hour_sin",
            "hour_cos",
            "dow_sin",
            "dow_cos",
            "doy_sin",
            "doy_cos"
        ]
    ].head()
)

(99288, 91)
            timestamp  hour_sin  hour_cos   dow_sin   dow_cos   doy_sin  \
0 2025-01-01 00:00:00  0.000000  1.000000  0.974928 -0.222521  0.017213   
1 2025-01-01 00:05:00  0.021815  0.999762  0.974928 -0.222521  0.017213   
2 2025-01-01 00:10:00  0.043619  0.999048  0.974928 -0.222521  0.017213   
3 2025-01-01 00:15:00  0.065403  0.997859  0.974928 -0.222521  0.017213   
4 2025-01-01 00:20:00  0.087156  0.996195  0.974928 -0.222521  0.017213   

    doy_cos  
0  0.999852  
1  0.999852  
2  0.999852  
3  0.999852  
4  0.999852  


In [79]:
lags = [1, 6, 12, 24, 72, 144, 288]

for lag in lags:
    df_fe[f"pv_lag_{lag}"] = df_fe["pv_power"].shift(lag)

print(df_fe.shape)

(99288, 98)


**Rolling Features**

In [80]:
# Rolling window sizes
windows = {
    "30m": 6,
    "1h": 12,
    "2h": 24,
    "6h": 72,
    "24h": 288
}

for name, window in windows.items():

    df_fe[f"pv_roll_mean_{name}"] = (
        df_fe["pv_power"]
        .rolling(window=window)
        .mean()
    )

    df_fe[f"pv_roll_std_{name}"] = (
        df_fe["pv_power"]
        .rolling(window=window)
        .std()
    )

print("Shape:", df_fe.shape)

Shape: (99288, 108)


In [81]:
missing = df_fe.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:", len(missing))
print(missing)

Columns with missing values: 17
pv_lag_288          288
pv_roll_std_24h     287
pv_roll_mean_24h    287
pv_lag_144          144
pv_lag_72            72
pv_roll_std_6h       71
pv_roll_mean_6h      71
pv_lag_24            24
pv_roll_mean_2h      23
pv_roll_std_2h       23
pv_lag_12            12
pv_roll_std_1h       11
pv_roll_mean_1h      11
pv_lag_6              6
pv_roll_std_30m       5
pv_roll_mean_30m      5
pv_lag_1              1
dtype: int64


In [82]:
df_fe = df_fe.iloc[288:].copy()

df_fe = df_fe.reset_index(drop=True)

print("Shape after removing initial rows:", df_fe.shape)

Shape after removing initial rows: (99000, 108)


In [83]:
missing = df_fe.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Remaining missing columns:", len(missing))
print(missing)

Remaining missing columns: 0
Series([], dtype: int64)


In [84]:
print("Total missing values:", df_fe.isna().sum().sum())

Total missing values: 0


Chronological Train / Validation / Test Split

In [85]:
# Total number of rows
n = len(df_fe)

# 70% Train
train_end = int(n * 0.70)

# 15% Validation
val_end = int(n * 0.85)

# Chronological split
train_df = df_fe.iloc[:train_end].copy()
val_df   = df_fe.iloc[train_end:val_end].copy()
test_df  = df_fe.iloc[val_end:].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (69300, 108)
Validation shape: (14850, 108)
Test shape: (14850, 108)


In [86]:
# Date range check
print("TRAIN")
print("Start:", train_df["timestamp"].min())
print("End  :", train_df["timestamp"].max())

print("\nVALIDATION")
print("Start:", val_df["timestamp"].min())
print("End  :", val_df["timestamp"].max())

print("\nTEST")
print("Start:", test_df["timestamp"].min())
print("End  :", test_df["timestamp"].max())

TRAIN
Start: 2025-01-02 00:00:00
End  : 2025-09-20 15:55:00

VALIDATION
Start: 2025-09-20 16:00:00
End  : 2025-11-11 10:45:00

TEST
Start: 2025-11-11 10:50:00
End  : 2026-01-02 00:50:00


In [87]:
print(
    "Train-Val overlap:",
    set(train_df["timestamp"]).intersection(
        set(val_df["timestamp"])
    ).__len__()
)

print(
    "Val-Test overlap:",
    set(val_df["timestamp"]).intersection(
        set(test_df["timestamp"])
    ).__len__()
)

Train-Val overlap: 0
Val-Test overlap: 0


In [88]:
print("\nTotal rows:")
print("Original:", len(df_fe))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print(
    "\nCheck:",
    len(train_df) + len(val_df) + len(test_df) == len(df_fe)
)


Total rows:
Original: 99000
Train: 69300
Validation: 14850
Test: 14850

Check: True


Scaling

In [89]:
from sklearn.preprocessing import StandardScaler

target = "pv_power"

feature_cols = [
    col for col in train_df.columns
    if col not in ["timestamp", target]
]

X_train = train_df[feature_cols].copy()
X_val   = val_df[feature_cols].copy()
X_test  = test_df[feature_cols].copy()

y_train = train_df[target].copy()
y_val   = val_df[target].copy()
y_test  = test_df[target].copy()

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

X_train: (69300, 106)
X_val  : (14850, 106)
X_test : (14850, 106)
y_train: (69300,)
y_val  : (14850,)
y_test : (14850,)


In [90]:
feature_scaler = StandardScaler()

X_train_scaled = feature_scaler.fit_transform(X_train)

X_val_scaled = feature_scaler.transform(X_val)

X_test_scaled = feature_scaler.transform(X_test)

print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled  :", X_val_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)

X_train_scaled: (69300, 106)
X_val_scaled  : (14850, 106)
X_test_scaled : (14850, 106)


In [91]:
target_scaler = StandardScaler()

y_train_scaled = target_scaler.fit_transform(
    y_train.to_numpy().reshape(-1, 1)
).ravel()

y_val_scaled = target_scaler.transform(
    y_val.to_numpy().reshape(-1, 1)
).ravel()

y_test_scaled = target_scaler.transform(
    y_test.to_numpy().reshape(-1, 1)
).ravel()

print("y_train_scaled:", y_train_scaled.shape)
print("y_val_scaled  :", y_val_scaled.shape)
print("y_test_scaled :", y_test_scaled.shape)

y_train_scaled: (69300,)
y_val_scaled  : (14850,)
y_test_scaled : (14850,)


In [92]:
print("Train feature mean:", X_train_scaled.mean())
print("Train feature std :", X_train_scaled.std())

print("Train target mean:", y_train_scaled.mean())
print("Train target std :", y_train_scaled.std())

Train feature mean: -2.731957018600487e-14
Train feature std : 1.0000000000000004
Train target mean: -1.271389599339862e-16
Train target std : 1.0


In [93]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

X_train: (69300, 106)
X_val: (14850, 106)
X_test: (14850, 106)
X_train_scaled: (69300, 106)
X_val_scaled: (14850, 106)
X_test_scaled: (14850, 106)


**Sequence / Window Creation**

In [94]:
# 5-minute interval

INPUT_WINDOW = 288   # 24 hours = 288 × 5 minutes

HORIZONS = {
    "30m": 6,      # 30 minutes
    "2h": 24,      # 2 hours
    "6h": 72,      # 6 hours
    "12h": 144,    # 12 hours
    "24h": 288     # 24 hours
}

print("Input window:", INPUT_WINDOW)
print("Horizons:", HORIZONS)

Input window: 288
Horizons: {'30m': 6, '2h': 24, '6h': 72, '12h': 144, '24h': 288}


In [95]:
print("Before dropna:", X_train_scaled.shape)

train_valid = ~np.isnan(X_train_scaled).any(axis=1)

X_train_clean = X_train_scaled[train_valid]
y_train_clean = y_train_scaled[train_valid]

print("After dropna:")
print("X:", X_train_clean.shape)
print("y:", y_train_clean.shape)

Before dropna: (69300, 106)
After dropna:
X: (69300, 106)
y: (69300,)


In [96]:
val_valid = ~np.isnan(X_val_scaled).any(axis=1)

X_val_clean = X_val_scaled[val_valid]
y_val_clean = y_val_scaled[val_valid]

print("Validation X:", X_val_clean.shape)
print("Validation y:", y_val_clean.shape)

Validation X: (14850, 106)
Validation y: (14850,)


In [97]:
test_valid = ~np.isnan(X_test_scaled).any(axis=1)

X_test_clean = X_test_scaled[test_valid]
y_test_clean = y_test_scaled[test_valid]

print("Test X:", X_test_clean.shape)
print("Test y:", y_test_clean.shape)

Test X: (14850, 106)
Test y: (14850,)


In [98]:
# Multi-Horizon Sequence Function
import numpy as np

def create_multi_horizon_sequences(X, y, input_window=288):

    X_seq = []
    y_seq = []

    max_horizon = max(HORIZONS.values())

    for i in range(input_window, len(X) - max_horizon + 1):

        # Past 24 hours
        X_seq.append(
            X[i-input_window:i]
        )

        # Future targets
        targets = [
            y[i + HORIZONS["30m"] - 1],
            y[i + HORIZONS["2h"] - 1],
            y[i + HORIZONS["6h"] - 1],
            y[i + HORIZONS["12h"] - 1],
            y[i + HORIZONS["24h"] - 1]
        ]

        y_seq.append(targets)

    return np.array(X_seq), np.array(y_seq)

In [99]:
print("X_train_clean shape:", X_train_clean.shape)
print("dtype:", X_train_clean.dtype)

print(
    "X_train_clean memory:",
    X_train_clean.nbytes / (1024**3),
    "GB"
)

X_train_clean shape: (69300, 106)
dtype: float64
X_train_clean memory: 0.05473047494888306 GB


In [100]:
import tensorflow as tf
import numpy as np

INPUT_WINDOW = 288

HORIZONS = {
    "30m": 6,
    "2h": 24,
    "6h": 72,
    "12h": 144,
    "24h": 288
}

max_horizon = max(HORIZONS.values())


def make_dataset(X, y, window=288, batch_size=64):

    X = X.astype(np.float32)
    y = y.astype(np.float32)

    n = len(X) - window - max_horizon + 1

    def generator():

        for i in range(n):

            X_window = X[i:i + window]

            y_future = np.array([
                y[i + window + HORIZONS["30m"] - 1],
                y[i + window + HORIZONS["2h"] - 1],
                y[i + window + HORIZONS["6h"] - 1],
                y[i + window + HORIZONS["12h"] - 1],
                y[i + window + HORIZONS["24h"] - 1]
            ], dtype=np.float32)

            yield X_window, y_future

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(
                shape=(window, X.shape[1]),
                dtype=tf.float32
            ),
            tf.TensorSpec(
                shape=(5,),
                dtype=tf.float32
            )
        )
    )

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset, n

In [101]:
train_dataset, train_samples = make_dataset(
    X_train_clean,
    y_train_clean,
    window=INPUT_WINDOW,
    batch_size=64
)

print("Train samples:", train_samples)

Train samples: 68725


In [102]:
val_dataset, val_samples = make_dataset(
    X_val_clean,
    y_val_clean,
    window=INPUT_WINDOW,
    batch_size=64
)

print("Validation samples:", val_samples)

Validation samples: 14275


In [103]:
test_dataset, test_samples = make_dataset(
    X_test_clean,
    y_test_clean,
    window=INPUT_WINDOW,
    batch_size=64
)

print("Test samples:", test_samples)

Test samples: 14275


In [104]:
for X_batch, y_batch in train_dataset.take(1):

    print("X batch shape:", X_batch.shape)
    print("y batch shape:", y_batch.shape)

X batch shape: (64, 288, 106)
y batch shape: (64, 5)


**model **

## TFT → Learns patterns from all features.
# N-BEATS → Learns patterns from the historical pv_power values.
# Adaptive Gate → Learns how much importance to give to each model’s prediction.
# Fusion → Combines the predictions from both models.
# Output → Predicts PV power for 30 minutes, 2 hours, 6 hours, 12 hours, and 24 hours.

```Input
(288 time steps, 106 features)
          │
     ┌────┴────┐
     ↓         ↓
   TFT      N-BEATS
     │         │
     │         │
     └────┬────┘
          ↓
   Adaptive Gate
          ↓
      Fusion
          ↓
   5 Predictions
          ↓
30m | 2h | 6h | 12h | 24h

```



In [133]:
import tensorflow as tf
from tensorflow.keras import layers, Model

N_FEATURES = 106
WINDOW = 288
N_HORIZONS = 5


# ============================================================
# 1. TFT Branch (multivariate, recency-aware pooling with fixed)
# ============================================================
def build_tft_branch(window=WINDOW, n_features=N_FEATURES, name="tft"):
    inp = layers.Input(shape=(window, n_features), name=f"{name}_input")

    x = layers.Dense(128, activation="relu", name=f"{name}_proj")(inp)
    x = layers.LayerNormalization(name=f"{name}_ln1")(x)

    x = layers.LSTM(128, return_sequences=True, name=f"{name}_lstm")(x)

    attn = layers.MultiHeadAttention(
        num_heads=4, key_dim=32, dropout=0.1, name=f"{name}_mha"
    )(x, x)
    x = layers.Add(name=f"{name}_add1")([x, attn])
    x = layers.LayerNormalization(name=f"{name}_ln2")(x)

    ffn = layers.Dense(256, activation="relu", name=f"{name}_ffn1")(x)
    ffn = layers.Dropout(0.1, name=f"{name}_drop1")(ffn)
    ffn = layers.Dense(128, name=f"{name}_ffn2")(ffn)
    x = layers.Add(name=f"{name}_add2")([x, ffn])
    x = layers.LayerNormalization(name=f"{name}_ln3")(x)

    # Recency-aware pooling: end timestep + overall average
    last_step = layers.Lambda(lambda t: t[:, -1, :], name=f"{name}_last_step")(x)
    avg_pool = layers.GlobalAveragePooling1D(name=f"{name}_avg_pool")(x)
    pooled = layers.Concatenate(name=f"{name}_pool_concat")([last_step, avg_pool])

    z = layers.Dense(128, activation="relu", name=f"{name}_dense_out")(pooled)
    z = layers.Dropout(0.1, name=f"{name}_drop_out")(z)

    return inp, z   # z = TFT embedding (also use fusion gate )


# ============================================================
# 2. N-BEATS Branch (univariate target history)
# ============================================================
def nbeats_block(x, units, window, horizon, name):
    h = layers.Dense(units, activation="relu", name=f"{name}_fc1")(x)
    h = layers.Dense(units, activation="relu", name=f"{name}_fc2")(h)
    h = layers.Dense(units, activation="relu", name=f"{name}_fc3")(h)
    h = layers.Dense(units, activation="relu", name=f"{name}_fc4")(h)

    backcast = layers.Dense(window, activation="linear", name=f"{name}_backcast")(h)
    forecast = layers.Dense(horizon, activation="linear", name=f"{name}_forecast")(h)
    return backcast, forecast


def build_nbeats_branch(window=WINDOW, horizon=N_HORIZONS,
                         n_blocks=3, units=256, name="nbeats"):
    inp = layers.Input(shape=(window,), name=f"{name}_input")

    residual = inp
    forecast_sum = None

    for i in range(n_blocks):
        backcast, forecast = nbeats_block(
            residual, units, window, horizon, name=f"{name}_block{i}"
        )
        residual = layers.Subtract(name=f"{name}_residual{i}")([residual, backcast])
        forecast_sum = forecast if forecast_sum is None else \
            layers.Add(name=f"{name}_forecast_add{i}")([forecast_sum, forecast])

    return inp, forecast_sum   # forecast_sum shape: (n_horizons,)


# ============================================================
# 3. Adaptive Gated Fusion
# ============================================================
def build_fusion_model(window=WINDOW, n_features=N_FEATURES, n_horizons=N_HORIZONS):

    tft_input, tft_embedding = build_tft_branch(window, n_features)
    tft_output = layers.Dense(n_horizons, name="tft_output")(tft_embedding)

    nbeats_input, nbeats_output = build_nbeats_branch(window, n_horizons)

    # Gate Network: The model learns a separate mixing weight for each forecasting horizon based on the information from both branches.
    gate_input = layers.Concatenate(name="gate_concat")([tft_embedding, nbeats_output])
    gate = layers.Dense(64, activation="relu", name="gate_dense1")(gate_input)
    gate = layers.Dense(
        n_horizons, activation="sigmoid", name="fusion_gate"
    )(gate)   # A separate weight is assigned to each horizon, with values ranging from 0 to 1

    weighted_tft = layers.Multiply(name="weighted_tft")([gate, tft_output])
    one_minus_gate = layers.Lambda(lambda g: 1.0 - g, name="one_minus_gate")(gate)
    weighted_nbeats = layers.Multiply(name="weighted_nbeats")([one_minus_gate, nbeats_output])

    fused_output = layers.Add(name="fused_output")([weighted_tft, weighted_nbeats])

    model = Model(
        inputs=[tft_input, nbeats_input],
        outputs=fused_output,
        name="TFT_NBEATS_AdaptiveGatedFusion"
    )
    return model


fusion_model = build_fusion_model()
fusion_model.summary()

Model: "TFT_NBEATS_AdaptiveGatedFusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ nbeats_input        │ (None, 288)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_block0_fc1   │ (None, 256)       │     73,984 │ nbeats_input[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_block0_fc2   │ (None, 256)       │     65,792 │ nbeats_block0_fc… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tft_input           │ (None, 288, 106)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_block0_fc3   │ (None, 256)       │     65,792 │ nbeats_block0_fc… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tft_proj (Dense)    │ (None, 288, 128)  │     13,696 │ tft_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_block0_fc4   │ (None, 256)       │     65,792 │ nbeats_block0_fc… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tft_ln1             │ (None, 288, 128)  │        256 │ tft_proj[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_block0_back… │ (None, 288)       │     74,016 │ nbeats_block0_fc… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tft_lstm (LSTM)     │ (None, 288, 128)  │    131,584 │ tft_ln1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_residual0    │ (None, 288)       │          0 │ nbeats_input[0][… │
│ (Subtract)          │                   │            │ nbeats_block0_ba… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tft_mha             │ (None, 288, 128)  │     66,048 │ tft_lstm[0][0],   │
│ (MultiHeadAttentio… │                   │            │ tft_lstm[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_block1_fc1   │ (None, 256)       │     73,984 │ nbeats_residual0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tft_add1 (Add)      │ (None, 288, 128)  │          0 │ tft_lstm[0][0],   │
│                     │                   │            │ tft_mha[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_block1_fc2   │ (None, 256)       │     65,792 │ nbeats_block1_fc… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tft_ln2             │ (None, 288, 128)  │        256 │ tft_add1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nbeats_block1_fc3   │ (None, 256)       │     65,792 │ nbeats_block1_fc… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 1,286,425 (4.91 MB)

 Trainable params: 1,286,425 (4.91 MB)

 Non-trainable params: 0 (0.00 B)

In [134]:
import numpy as np
import tensorflow as tf

INPUT_WINDOW = 288

HORIZONS = {
    "30m": 6,
    "2h": 24,
    "6h": 72,
    "12h": 144,
    "24h": 288
}

max_horizon = max(HORIZONS.values())


def make_fusion_dataset(X, y, window=288, batch_size=64, shuffle=False):

    X = X.astype(np.float32)
    y = y.astype(np.float32)

    n = len(X) - window - max_horizon + 1

    def generator():
        for i in range(n):
            X_window = X[i:i + window]        # TFT input: (window, n_features)
            y_window = y[i:i + window]         # N-BEATS input: (window,)

            y_future = np.array([
                y[i + window + HORIZONS["30m"] - 1],
                y[i + window + HORIZONS["2h"] - 1],
                y[i + window + HORIZONS["6h"] - 1],
                y[i + window + HORIZONS["12h"] - 1],
                y[i + window + HORIZONS["24h"] - 1]
            ], dtype=np.float32)

            yield (X_window, y_window), y_future

    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            (
                tf.TensorSpec(shape=(window, X.shape[1]), dtype=tf.float32),
                tf.TensorSpec(shape=(window,), dtype=tf.float32),
            ),
            tf.TensorSpec(shape=(5,), dtype=tf.float32)
        )
    )

    if shuffle:
        dataset = dataset.shuffle(buffer_size=2000)

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset, n


# Create the Train/Validation/Test datasets using the previously cleaned data.
train_fusion_dataset, train_samples = make_fusion_dataset(
    X_train_clean, y_train_clean, window=INPUT_WINDOW, batch_size=64, shuffle=True
)

val_fusion_dataset, val_samples = make_fusion_dataset(
    X_val_clean, y_val_clean, window=INPUT_WINDOW, batch_size=64
)

test_fusion_dataset, test_samples = make_fusion_dataset(
    X_test_clean, y_test_clean, window=INPUT_WINDOW, batch_size=64
)

print("Train samples:", train_samples)
print("Val samples:  ", val_samples)
print("Test samples: ", test_samples)

# Shape verify
for (X_b, y_hist_b), y_fut_b in train_fusion_dataset.take(1):
    print("TFT input shape:    ", X_b.shape)
    print("N-BEATS input shape:", y_hist_b.shape)
    print("Target shape:       ", y_fut_b.shape)

Train samples: 68725
Val samples:   14275
Test samples:  14275
TFT input shape:     (64, 288, 106)
N-BEATS input shape: (64, 288)
Target shape:        (64, 5)


In [135]:
from tensorflow.keras.optimizers import Adam

# Take a small batch.
for (X_small, y_hist_small), y_small in train_fusion_dataset.take(1):
    X_small = X_small.numpy()
    y_hist_small = y_hist_small.numpy()
    y_small = y_small.numpy()
    break

small_model = build_fusion_model()
small_model.compile(optimizer=Adam(learning_rate=1e-3), loss="mse")

hist = small_model.fit(
    [X_small, y_hist_small], y_small,
    epochs=300, verbose=0
)
print("Final loss on small batch:", hist.history["loss"][-1])
# Expected: It should drop below 0.01. If it does not, there may be a wiring bug.

Final loss on small batch: 0.0011557848192751408


In [136]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import time

fusion_model = build_fusion_model()

fusion_model.compile(
    optimizer=Adam(learning_rate=1e-3, clipnorm=1.0),
    loss="mse",
    metrics=["mae"]
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1
)
early_stop = EarlyStopping(
    monitor="val_loss", patience=6, restore_best_weights=True, verbose=1
)

start_time = time.time()

history = fusion_model.fit(
    train_fusion_dataset,
    validation_data=val_fusion_dataset,
    epochs=30,
    callbacks=[reduce_lr, early_stop]
)

elapsed = time.time() - start_time
print(f"\nFusion Model Training Time: {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")

Epoch 1/30
1074/1074 ━━━━━━━━━━━━━━━━━━━━ 97s 78ms/step - loss: 0.0468 - mae: 0.1122 - val_loss: 0.4684 - val_mae: 0.3855 - learning_rate: 0.0010
Epoch 2/30
1074/1074 ━━━━━━━━━━━━━━━━━━━━ 83s 77ms/step - loss: 0.0428 - mae: 0.0993 - val_loss: 0.3838 - val_mae: 0.3451 - learning_rate: 0.0010
Epoch 3/30
1074/1074 ━━━━━━━━━━━━━━━━━━━━ 141s 76ms/step - loss: 0.0390 - mae: 0.0947 - val_loss: 0.3851 - val_mae: 0.3318 - learning_rate: 0.0010
Epoch 4/30
1074/1074 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.0479 - mae: 0.1067
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
1074/1074 ━━━━━━━━━━━━━━━━━━━━ 81s 75ms/step - loss: 0.0384 - mae: 0.0916 - val_loss: 0.4871 - val_mae: 0.3555 - learning_rate: 0.0010
Epoch 5/30
1074/1074 ━━━━━━━━━━━━━━━━━━━━ 82s 76ms/step - loss: 0.0335 - mae: 0.0815 - val_loss: 0.4021 - val_mae: 0.3283 - learning_rate: 5.0000e-04
Epoch 6/30
1074/1074 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 0.0403 - mae: 0.0931
Epoch 6: ReduceLROnPlateau reduc

In [145]:
# parameter count and model size
def get_model_stats(model, name="Model"):
    total_params = model.count_params()
    trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])

    # Model size (approx, float32 = 4 bytes per param)
    size_mb = (total_params * 4) / (1024 ** 2)

    print(f"=== {name} ===")
    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Approx model size:    {size_mb:.2f} MB")
    print()

    return total_params, size_mb

# Standalone TFT model (the previous one)
tft_only_model = build_tft_model()
tft_params, tft_size = get_model_stats(tft_only_model, "TFT Only")

# Fusion model
fusion_params, fusion_size = get_model_stats(fusion_model, "TFT + N-BEATS Fusion")

print(f"Fusion model is {fusion_params/tft_params:.2f}x larger than TFT-only")

=== TFT Only ===
Total parameters:     295,173
Trainable parameters: 295,173
Approx model size:    1.13 MB

=== TFT + N-BEATS Fusion ===
Total parameters:     1,286,425
Trainable parameters: 1,286,425
Approx model size:    4.91 MB

Fusion model is 4.36x larger than TFT-only


In [137]:
# Per-horizon evaluation
# Make predictions on the validation set
preds = fusion_model.predict(val_fusion_dataset)
y_true = np.concatenate([y for _, y in val_fusion_dataset], axis=0)

horizon_names = list(HORIZONS.keys())

print("Per-horizon Validation MSE:")
for h_idx, h_name in enumerate(horizon_names):
    mse_h = np.mean((preds[:, h_idx] - y_true[:, h_idx]) ** 2)
    mae_h = np.mean(np.abs(preds[:, h_idx] - y_true[:, h_idx]))
    print(f"  {h_name:>5}: MSE = {mse_h:.4f}  MAE = {mae_h:.4f}")

# Compare it with the naive baseline.
naive_mse = np.mean((y_true - 0.0) ** 2)  # scaled zero = night baseline
print(f"\nOverall Naive MSE (predict 0): {naive_mse:.4f}")
print(f"Overall Fusion Model MSE:      {np.mean((preds - y_true)**2):.4f}")

# Day-only MSE (excluding trivial nighttime zero values).
day_mask = y_true[:, 0] > -0.7
print(f"\nDay-only samples: {day_mask.sum()} / {len(day_mask)}")
print(f"Day-only Fusion MSE: {np.mean((preds[day_mask] - y_true[day_mask])**2):.4f}")

224/224 ━━━━━━━━━━━━━━━━━━━━ 73s 325ms/step
Per-horizon Validation MSE:
    30m: MSE = 0.2535  MAE = 0.2624
     2h: MSE = 0.2388  MAE = 0.2624
     6h: MSE = 0.4394  MAE = 0.3578
    12h: MSE = 0.4667  MAE = 0.4095
    24h: MSE = 0.5207  MAE = 0.4335

Overall Naive MSE (predict 0): 1.3302
Overall Fusion Model MSE:      0.3838

Day-only samples: 7539 / 14275
Day-only Fusion MSE: 0.5161


In [138]:
# Check interpretability by examining the gate values
gate_model = Model(
    inputs=fusion_model.inputs,
    outputs=fusion_model.get_layer("fusion_gate").output
)

all_gates = []
for (X_b, y_hist_b), y_fut_b in val_fusion_dataset:
    g = gate_model.predict([X_b, y_hist_b], verbose=0)
    all_gates.append(g)

all_gates = np.concatenate(all_gates, axis=0)

print("Average gate value per horizon (গড়, 1.0 = full TFT, 0.0 = full N-BEATS):")
for h_idx, h_name in enumerate(horizon_names):
    print(f"  {h_name:>5}: {all_gates[:, h_idx].mean():.3f}")

Average gate value per horizon (গড়, 1.0 = full TFT, 0.0 = full N-BEATS):
    30m: 0.617
     2h: 0.607
     6h: 0.687
    12h: 0.656
    24h: 0.655


In [139]:
# Examine the variability of gate values across each forecasting horizon
print("Gate value std per horizon (sample-to-sample variation):")
for h_idx, h_name in enumerate(horizon_names):
    print(f"  {h_name:>5}: mean={all_gates[:, h_idx].mean():.3f}  "
          f"std={all_gates[:, h_idx].std():.3f}  "
          f"min={all_gates[:, h_idx].min():.3f}  "
          f"max={all_gates[:, h_idx].max():.3f}")

Gate value std per horizon (sample-to-sample variation):
    30m: mean=0.617  std=0.126  min=0.422  max=0.820
     2h: mean=0.607  std=0.131  min=0.358  max=0.824
     6h: mean=0.687  std=0.109  min=0.479  max=0.900
    12h: mean=0.656  std=0.074  min=0.388  max=0.806
    24h: mean=0.655  std=0.104  min=0.357  max=0.829


In [141]:
# 1. Day vs Night gate comparison
y_true = np.concatenate([y for _, y in val_fusion_dataset], axis=0)
day_mask = y_true[:, 0] > -0.7

print("Gate value — Day vs Night comparison:")
for h_idx, h_name in enumerate(horizon_names):
    day_gate = all_gates[day_mask, h_idx].mean()
    night_gate = all_gates[~day_mask, h_idx].mean()
    print(f"  {h_name:>5}: Day={day_gate:.3f}  Night={night_gate:.3f}  (diff={day_gate-night_gate:+.3f})")

# 2.Per-horizon MSE — final actual performance evaluation.
preds = fusion_model.predict(val_fusion_dataset)

print("\nPer-horizon Validation MSE (Fusion Model):")
for h_idx, h_name in enumerate(horizon_names):
    mse_h = np.mean((preds[:, h_idx] - y_true[:, h_idx]) ** 2)
    print(f"  {h_name:>5}: MSE = {mse_h:.4f}")

overall_mse = np.mean((preds - y_true) ** 2)
naive_mse = np.mean((y_true - (-0.727)) ** 2)

print(f"\nOverall Fusion Model MSE: {overall_mse:.4f}")
print(f"Naive baseline MSE (always predict night-zero): {naive_mse:.4f}")
print(f"Improvement over naive: {(1 - overall_mse/naive_mse)*100:.1f}%")

# 3.Day-only MSE (actual forecasting performance, excluding trivial nighttime zero values).
day_mse = np.mean((preds[day_mask] - y_true[day_mask]) ** 2)
print(f"\nDay-only Fusion MSE: {day_mse:.4f}")

Gate value — Day vs Night comparison:
    30m: Day=0.512  Night=0.735  (diff=-0.222)
     2h: Day=0.510  Night=0.716  (diff=-0.206)
     6h: Day=0.634  Night=0.746  (diff=-0.112)
    12h: Day=0.642  Night=0.672  (diff=-0.030)
    24h: Day=0.588  Night=0.730  (diff=-0.142)
224/224 ━━━━━━━━━━━━━━━━━━━━ 7s 31ms/step

Per-horizon Validation MSE (Fusion Model):
    30m: MSE = 0.2535
     2h: MSE = 0.2388
     6h: MSE = 0.4394
    12h: MSE = 0.4667
    24h: MSE = 0.5207

Overall Fusion Model MSE: 0.3838
Naive baseline MSE (always predict night-zero): 2.1760
Improvement over naive: 82.4%

Day-only Fusion MSE: 0.5161


In [144]:
# Day-only naive baseline (performance when predicting the daytime mean)
day_y_true = y_true[day_mask]
day_naive_mean = day_y_true.mean()
day_naive_mse = np.mean((day_y_true - day_naive_mean) ** 2)

print(f"Day-only naive baseline MSE (predict day-mean): {day_naive_mse:.4f}")
print(f"Day-only Fusion MSE:                            0.5161")
print(f"Day-only improvement: {(1 - 0.5161/day_naive_mse)*100:.1f}%")

Day-only naive baseline MSE (predict day-mean): 1.3453
Day-only Fusion MSE:                            0.5161
Day-only improvement: 61.6%


In [146]:
# Inference Time (latency) Compare
import time
import numpy as np

def measure_inference_time(model, inputs, n_runs=100, warmup=10):
    # Warm-up runs (to exclude GPU/graph compilation overhead)
    for _ in range(warmup):
        _ = model.predict(inputs, verbose=0)

    start = time.time()
    for _ in range(n_runs):
        _ = model.predict(inputs, verbose=0)
    elapsed = time.time() - start

    avg_time_ms = (elapsed / n_runs) * 1000
    return avg_time_ms

# Take one batch (batch_size = 64) for comparison.
for (X_b, y_hist_b), y_fut_b in val_fusion_dataset.take(1):
    X_sample = X_b.numpy()
    y_hist_sample = y_hist_b.numpy()
    break

# TFT-only inference time
tft_only_time = measure_inference_time(tft_only_model, X_sample)
print(f"TFT-only avg inference time (batch=64): {tft_only_time:.2f} ms")

# Fusion model inference time
fusion_time = measure_inference_time(fusion_model, [X_sample, y_hist_sample])
print(f"Fusion model avg inference time (batch=64): {fusion_time:.2f} ms")

print(f"\nFusion model is {fusion_time/tft_only_time:.2f}x slower than TFT-only")

# Single-sample (real-time deployment scenario) latency
X_single = X_sample[:1]
y_hist_single = y_hist_sample[:1]

tft_single_time = measure_inference_time(tft_only_model, X_single)
fusion_single_time = measure_inference_time(fusion_model, [X_single, y_hist_single])

print(f"\nSingle-sample latency:")
print(f"  TFT-only: {tft_single_time:.2f} ms")
print(f"  Fusion:   {fusion_single_time:.2f} ms")

TFT-only avg inference time (batch=64): 172.92 ms
Fusion model avg inference time (batch=64): 120.99 ms

Fusion model is 0.70x slower than TFT-only

Single-sample latency:
  TFT-only: 83.93 ms
  Fusion:   89.96 ms
